# 🏷️ 价格合适 — 第 7 周练习

## 通过平衡价格桶采样改进 QLoRA 微调

**作者：Vagz1216 (Haqs12)**

---

### 我正在解决的问题

在讲师的第 3/4 天培训笔记本中，培训数据直接从
没有过滤的“ed-donner/items_prompts_lite”——20,000 种亚马逊产品的随机切片。

问题在于亚马逊严重偏向廉价商品。 20,000 个随机样本
产品将包含比“400 美元电动工具”更多的“5 美元手机壳”和“12 美元电缆”或
“800美元的电器”。当微调模型在这些不平衡数据上进行训练时，它学会了
善于猜测便宜的价格——但它见过的昂贵物品太少了，所以它
一直低估他们。

### 我的解决方案：价格桶平衡的训练数据

在第 6 周的练习中，我为 OpenAI 微调构建了**价格桶平衡算法**。
我将同样的想法应用到 QLoRA 开源培训管道中。

我将训练数据分为 4 个价格桶，并从每个价格桶中**平等**地进行采样：

|桶|价格范围 |已选项目 |
|--------|-------------|----------------|
|预算| \$0 – \$50 |训练集的 25% |
|中档| $50 – $150 |训练集的 25% |
|高级| $150 – $300 |训练集的 25% |
|豪华| $300+ |训练集的 25% |

假设是：**在平衡价格分布上训练的模型会赚更多
整个价格范围**的准确预测，而不仅仅是廉价商品。

> 🖥️ **使用免费的 T4 GPU 在 Google Colab 中运行。**  
> 在运行之前将您的“HF_TOKEN”和“WANDB_API_KEY”添加到 Colab Secrets 中。

---
## 步骤 1：安装库

In [ ]:
# 固定到与讲师第 3/4 天培训笔记本相同的版本
# Pinned to the same versions as the instructor's Day 3/4 training notebook
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

# 从课程存储库中提取 Ed 的评估（）和测试器实用程序
# Pulling Ed's evaluate() and Tester utilities from the course repo
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

print("Libraries installed!")

In [ ]:
import os
import re
import random
import math
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import wandb
import matplotlib.pyplot as plt
from util import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

---
## 步骤 2：常量

我保留“LITE_MODE = True”，因为我使用的是免费的 T4 GPU。
与讲师笔记本的主要区别是我不会**使用
直接预格式化“items_prompts_lite”数据集。
相反，我加载原始的“items_lite”数据集，以便我可以访问价格字段
并在重新格式化之前应用我自己的平衡采样。

In [ ]:
# 基础开源模型——与讲师相同
# Base open-source model — same as the instructor
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"

# 我加载原始项目数据集（带有价格字段）而不是预先格式化的提示
# I load the raw items dataset (with price field) instead of pre-formatted prompts
# 这样我就可以在格式化之前应用价格桶过滤
# so I can apply price-bucket filtering before formatting
DATA_USER = "ed-donner"
RAW_DATASET = f"{DATA_USER}/items_lite"    # Has .price, .summary, .title fields
PROMPT_DATASET = f"{DATA_USER}/items_prompts_lite"  # Used for test evaluation only

# 保持在 T4 GPU 限制内
# Keep within T4 GPU limits
LITE_MODE = True

# 我的 HuggingFace 用户名 — 运行时将其更新为您的用户名
# My HuggingFace username — update this to yours when running
HF_USER = "Haqs12"

# 跑步追踪
# Run tracking
# --- 训练/模型加载配置 ---
# --- Training / Model Loading Config ---
# 默认情况下，此笔记本动态运行新的训练作业：
# By default, this notebook runs a fresh training job dynamically:
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite-balanced"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# 💡 审稿人注意事项：如果您想跳过训练并评估精确模型
# 💡 NOTE FOR REVIEWERS: If you want to skip training and evaluate the EXACT model
# 在下面的结果中实现了 79.18 美元的平均误差，注释掉这 3 行
# that achieved the $79.18 average error in the results below, comment out the 3 lines
# 上面并取消注释下面的 3 行以直接加载我的微调快照：
# above and uncomment the 3 lines below to load my fine-tuned snapshot directly:
# RUN_NAME =“2026-03-11_11.38.39-lite-平衡”
# RUN_NAME = "2026-03-11_11.38.39-lite-balanced"
# PROJECT_RUN_NAME = f“{PROJECT_NAME}-{RUN_NAME}”
# PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# HUB_MODEL_NAME = "Haqs12/price-2026-03-11_11.38.39-lite-balanced"

# --- 训练超参数（与讲师的 LITE_MODE 设置相同）---
# --- Training Hyperparameters (same as instructor's LITE_MODE settings) ---
EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# --- QLoRA 超参数（与讲师的 LITE_MODE 设置相同）---
# --- QLoRA Hyperparameters (same as instructor's LITE_MODE settings) ---
QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = LORA_R * 2       # Standard convention: alpha = 2x rank
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS   # Targeting attention layers only (T4 memory limit)
LORA_DROPOUT = 0.1

# --- 优化器和 LR ---
# --- Optimizer & LR ---
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
OPTIMIZER = "paged_adamw_32bit"

# 检测GPU能力
# Detect GPU capability
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# 记录
# Logging
VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = True

print(f"GPU capability: {capability} → {'bfloat16' if use_bf16 else 'float16'}")
print(f"Model will be saved to HuggingFace as: {HUB_MODEL_NAME}")

---
## 第 3 步：登录

In [ ]:
# HuggingFace — 需要下载原始数据集并推送微调后的模型
# HuggingFace — needed to download the raw dataset and push the fine-tuned model
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace!")

# 权重和偏差——用于跟踪训练损失曲线
# Weights & Biases — for tracking training loss curves
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
print("Logged in to Weights & Biases!")

---
## 步骤 4：加载原始数据集并应用价格桶平衡抽样

这是核心的改进。不是向 QLoRA 训练器提供随机切片
在数据集中，我首先检查每个商品的价格，将它们分类到 4 个桶中，
并从每个桶中取出相同的数量。

“items_lite”数据集有一个“price”字段，我们可以直接用于存储。
讲师的“items_prompts_lite”跳过了这一点——它只有“提示”和“完成”
字符串，这使得基于价格的过滤变得更加困难。

然后，我使用相同的方法将平衡项目重新格式化为“提示”/“完成”对
教师使用的模板（“这个价格是多少，最接近的美元？...价格是美元”）。

In [ ]:
# 加载原始商品数据集——包含价格、摘要、标题、类别等。
# Load the raw items dataset — has price, summary, title, category etc.
raw = load_dataset(RAW_DATASET)
raw_train = raw['train']
raw_val = raw['validation']

print(f"Raw training items: {len(raw_train):,}")
print(f"Raw validation items: {len(raw_val):,}")
print("\nSample item fields:", list(raw_train[0].keys()))

In [ ]:
# --- 价格范围定义 ---
# --- Price Bucket Definitions ---
# 我在第 6 周的练习中构建了相同的 4 桶系统。
# The same 4-bucket system I built in my Week 6 exercise.
# 目标是强制廉价和昂贵物品的平等代表性。
# The goal is to force equal representation of cheap and expensive items.

def categorize_price(price):
    """Sort a price into one of 4 buckets."""
    if price < 50:
        return '$0-50'
    elif price < 150:
        return '$50-150'
    elif price < 300:
        return '$150-300'
    else:
        return '$300+'


def balance_by_price(dataset, items_per_bucket, seed=42):
    """
    Groups dataset items by price bucket and samples equally from each.
    Returns a shuffled, balanced list of HuggingFace dataset row dicts.
    """
    random.seed(seed)
    buckets = defaultdict(list)

    for item in dataset:
        price = item.get('price')
        # 跳过没有价格或无效价格的商品
        # Skip items with no price or invalid price
        if price is None or not isinstance(price, (int, float)) or price <= 0:
            continue
        bucket = categorize_price(price)
        buckets[bucket].append(item)

    print("Price distribution BEFORE balancing:")
    for b in sorted(buckets.keys()):
        print(f"  {b}: {len(buckets[b]):,} items")

    balanced = []
    for b in sorted(buckets.keys()):
        sample_size = min(items_per_bucket, len(buckets[b]))
        sample = random.sample(buckets[b], sample_size)
        balanced.extend(sample)
        print(f"  Selected {sample_size} items from {b} bucket")

    random.shuffle(balanced)  # Shuffle so prices aren't grouped during training
    return balanced


# 应用平衡 — 每个存储桶 1,000 个项目 = 4,000 个总训练项目
# Apply balancing — 1,000 items per bucket = 4,000 total training items
# 这仍然完全在 LITE_MODE 的 T4 内存限制之内
# This is still well within T4 memory limits for LITE_MODE
ITEMS_PER_BUCKET = 1000

print("\n--- Balancing Training Data ---")
balanced_train_items = balance_by_price(raw_train, items_per_bucket=ITEMS_PER_BUCKET)
print(f"\nFinal balanced training set: {len(balanced_train_items):,} items")

print("\n--- Balancing Validation Data ---")
balanced_val_items = balance_by_price(raw_val, items_per_bucket=125)  # 125 per bucket = 500 val items
print(f"Final balanced validation set: {len(balanced_val_items):,} items")

In [ ]:
# 可视化均衡的培训价格分布
# Visualise the balanced training price distribution
# 均衡分布应显示整个价格范围内的数量大致相等
# A balanced distribution should show roughly equal counts across the price range
train_prices = [item['price'] for item in balanced_train_items]

plt.figure(figsize=(12, 4))
plt.hist(train_prices, bins=50, color='steelblue', rwidth=0.8)
plt.title(f"Price Distribution of My Balanced Training Set ({len(train_prices):,} items)")
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Bucket boundaries')
plt.axvline(x=150, color='red', linestyle='--', alpha=0.7)
plt.axvline(x=300, color='red', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Price stats: min=${min(train_prices):.2f}, max=${max(train_prices):.2f}, "
      f"mean=${sum(train_prices)/len(train_prices):.2f}")

---
## 步骤 5：格式化为提示/完成对

现在我将平衡的“Item”字典转换为相同的“prompt”/“completion”字符串
教师的“SFTTrainer”期望的格式。

该模板与“items.py”生成的内容相同 - 因此模型将看到
其设计的文本结构相同，只是取自更平衡的文本
数据切片。

In [ ]:
# 这些常量与讲师在 week7/pricer/items.py 中定义的内容完全匹配
# These constants match exactly what the instructor defined in week7/pricer/items.py
QUESTION = "What does this cost to the nearest dollar?"
PREFIX = "Price is $"


def make_prompt_completion(item, tokenizer, max_tokens=128, include_answer=True):
    """
    Converts a raw item dict into the prompt/completion format expected by SFTTrainer.
    Mirrors the logic in the instructor's items.py make_prompts() method.
    """
    summary = item.get('summary') or item.get('title', '')

    # 将摘要截断为 max_tokens — 与讲师的 CUTOFF=110 的逻辑相同
    # Truncate summary to max_tokens — same logic as the instructor's CUTOFF=110
    tokens = tokenizer.encode(summary, add_special_tokens=False)
    if len(tokens) > max_tokens:
        summary = tokenizer.decode(tokens[:max_tokens]).rstrip()

    prompt = f"{QUESTION}\n\n{summary}\n\n{PREFIX}"

    if include_answer:
        # 培训/验证项目包括答案：“价格为 299.00 美元”
        # Training/validation items include the answer: "Price is $299.00"
        completion = f"{round(item['price'])}.00"
    else:
        # 测试项目以“Price is $”结尾——模型必须预测数量
        # Test items end at "Price is $" — model must predict the number
        completion = str(item['price'])

    return {"prompt": prompt, "completion": completion}


# 现在加载分词器，以便我们可以在格式化期间使用它进行截断
# Load the tokenizer now so we can use it for truncation during formatting
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Formatting balanced training data...")
train_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                   for item in tqdm(balanced_train_items)]

print("Formatting balanced validation data...")
val_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                 for item in tqdm(balanced_val_items)]

# 转换为 HuggingFace 数据集格式 — SFTTrainer 所期望的
# Convert to HuggingFace Dataset format — what SFTTrainer expects
train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"\nTraining dataset: {len(train_dataset):,} items")
print(f"Validation dataset: {len(val_dataset):,} items")
print("\nSample formatted training item:")
print(train_dataset[0]['prompt'])
print("Completion:", train_dataset[0]['completion'])

---
## 步骤 6：加载具有 4 位量化的基本模型

与讲师第 3/4 天的设置相同 — 以 4 位 NF4 精度加载 Llama 3.2 3B
使其符合 T4 的 16GB VRAM 预算。

In [ ]:
# 4 位量化配置 — 与讲师的设置相同
# 4-bit quantization config — identical to the instructor's setup
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )

# 加载冻结的基础模型
# Load the frozen base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Base model loaded! Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

---
## 步骤 7：配置 QLoRA 并训练

LoRA 和 SFT 配置与讲师的 LITE_MODE 设置相同。
唯一改变的是**我们输入的数据**——我的平衡集
而不是随机切片。这将数据质量的改进隔离为
本实验中的自变量。

In [ ]:
# LoRA 适配器配置 — 与讲师的 LITE_MODE 相同
# LoRA adapter configuration — same as instructor's LITE_MODE
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,    # Scaling factor for the adapter updates
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,                 # Rank of the low-rank decomposition (32 for LITE_MODE)
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,  # Only attention layers to save T4 memory
)

print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}")

In [ ]:
# 初始化 WandB 跟踪运行
# Initialise WandB tracking run
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# 训练配置 — 准确反映讲师的 LITE_MODE SFTConfig
# Training configuration — mirrors the instructor's LITE_MODE SFTConfig exactly
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,       # Auto-saves to HuggingFace every SAVE_STEPS — crash-safe!
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
)

print(f"Training config ready. Will push checkpoints to: {HUB_MODEL_NAME}")

In [ ]:
# 将所有内容连接到 SFTTrainer
# Wire everything together into the SFTTrainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,      # My price-balanced training data
    eval_dataset=val_dataset,          # My price-balanced validation data
    peft_config=lora_parameters,
    args=train_parameters
)

# 开始训练！
# KICK OFF TRAINING!
# 在包含 4,000 个项目的免费 T4 上，这大约需要 10-20 分钟
# On a free T4 with 4,000 items, this should take roughly 10-20 minutes
fine_tuning.train()

# 完成后将最终模型推送到 HuggingFace
# Push the final model to HuggingFace when done
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"\nTraining complete! Model saved to: {HUB_MODEL_NAME}")

if LOG_TO_WANDB:
    wandb.finish()

---
## 步骤 8：评估 — 平衡训练数据有帮助吗？

现在，我针对相同的 200 项测试集测试我的价格平衡微调模型
教师使用，直接将我的分数与基线进行比较。

In [ ]:
# 加载标准测试拆分以进行公平的同类比较
# Load the standard test split for a fair apples-to-apples comparison
# 与教师的基线模型分数
# with the instructor's baseline model score
prompt_ds = load_dataset(PROMPT_DATASET)
test = prompt_ds['test']
print(f"Test items: {len(test):,}")

In [ ]:
# 加载我新训练的模型进行评估
# Load my newly trained model for evaluation
# （如果您在崩溃后运行评估，请将 HUB_MODEL_NAME 更新到您的存储库）
# (If you're running evaluation after a crash, update HUB_MODEL_NAME to your repo)
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
print(f"Fine-tuned model loaded! Memory: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def model_predict(item):
    """
    Standard inference function — identical to the instructor's Day 5 model_predict().
    Using the same greedy decoding so any improvement in score comes purely from
    the better-balanced training data, not a decoding trick.
    """
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        with torch.autocast("cuda", dtype=torch.float16):
            output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


# 对 3 项物品进行健全性检查
# Sanity check on 3 items
print("Sanity check:")
for i in range(3):
    pred = model_predict(test[i]).strip()
    actual = test[i]['completion']
    print(f"  Item {i+1}: Predicted '{pred}' | Actual: ${actual}")


In [ ]:
# 对 200 个看不见的测试项目进行全面评估
# Full evaluation across 200 unseen test items
print("Running full evaluation (200 test items)...")
set_seed(42)
evaluate(model_predict, test)

---
## 结果与反思

|型号|培训数据|平均误差 ($) |
|---|---|---|
|教师的基线（随机样本）| `items_prompts_lite`（20,000 项）| 65.40 美元 |
|我的模型（价格平衡样本）| `items_lite` 按存储桶平衡（4,000 个项目）| 79.18 美元 |

### 我观察到的：

我的微调模型（仅在 4,000 个价格平衡的商品上进行训练）的平均误差为 **79.18 美元**。

结果表明，与讲师的精简版微调模型相比，我的模型表现较差（65.40 美元 vs 79.18 美元）。
然而，有一个极其重要的警告：**讲师的模型训练了 20,000 个项目** — 数据比我的多 5 倍。

尽管在较小的数据集上进行训练，我的模型仍然**轻松击败人类基线（87.62 美元）** 和 **Base Llama 3.2 4 位模型（110.72 美元）**！ （如导师综合评价表所示）。

该实验的主要见解是**数据质量很重要**。亚马逊数据的简单随机样本以廉价商品为主。通过强制训练集查看每个价格层中相同数量的商品，模型将接触到现实世界中更具代表性的版本。

随着时间的推移，我会尝试更多的方法来看到它表现得更好。例如，将这种平衡方法扩展到 20,000 个项目（这需要付费 GPU 来适应批量内存限制），我假设它将完全在昂贵的产品精度上击败随机样本基线。

这直接基于第 6 周的数据工程课程：
> *“数据管理可以被认为是数据科学家的一项不太光彩的工作。
> 我说那是废话！这就是科学发生的地方。”* — Ed Donner

---
_第 7 周顶点练习 |瓦格兹1216 | Andela人工智能工程训练营_